# Chương 5 – Fine-tune LoRA và QLoRA (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter05/chapter05_colab.ipynb)

Fine-tune `Qwen/Qwen3-4B-Instruct-2507` trên dữ liệu IT-support bằng LoRA (r=16) và QLoRA (4-bit, r=8),
đánh giá so với model gốc, kiểm tra safety regression và chạy inference với adapter.

**GPU:** L4 (24 GB) là đủ — LoRA cần ~9 GB, QLoRA ~5 GB. Nếu không muốn train, nhảy tới
**mục 6** để dùng adapter đã publish trên Hugging Face.

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

In [ ]:
import os, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev,qlora]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

## 1. Kiểm tra môi trường

In [ ]:
run("python chapter05/scripts/validate_chapter05.py")

## 2. Train LoRA (Listing 5.2)

450 ví dụ × 3 epoch — ~11 phút trên A30, ~15–25 phút trên L4.

In [ ]:
run("python -m chapter05.train_lora "
    "--train data/it_support_fmt/train.jsonl "
    "--valid data/it_support_fmt/valid.jsonl "
    "--out chapter05/runs/it_lora")

## 3. Đánh giá LoRA vs model gốc + safety regression (Listing 5.3 / 5.6)

~10–15 phút.

In [ ]:
run("python -m chapter05.scripts.listing_5_3_evaluate "
    "--base Qwen/Qwen3-4B-Instruct-2507 "
    "--adapter chapter05/runs/it_lora "
    "--test data/it_support/test.jsonl")

In [ ]:
import pathlib
from IPython.display import Markdown, display
rep = pathlib.Path("chapter05/runs/eval_report/report.md")
display(Markdown(rep.read_text())) if rep.exists() else print("Chưa có report.md")

## 4. Inference với adapter (Listing 5.4)

In [ ]:
run("python -m chapter05.generate "
    "--base Qwen/Qwen3-4B-Instruct-2507 "
    "--adapter chapter05/runs/it_lora "
    "--max_new_tokens 256 "
    "--prompt \"My laptop won't connect to the office Wi-Fi after a Windows update. How do I fix it?\"")

## 5. (Tuỳ chọn) QLoRA 4-bit (Listing 5.5) và so sánh LoRA vs QLoRA

~15–25 phút train + ~15 phút đánh giá.

In [ ]:
run("python -m chapter05.train_qlora "
    "--train data/it_support_fmt/train.jsonl "
    "--valid data/it_support_fmt/valid.jsonl "
    "--out chapter05/runs/it_qlora")

In [ ]:
run("python -m chapter05.scripts.listing_5_3_evaluate "
    "--base Qwen/Qwen3-4B-Instruct-2507 "
    "--adapter chapter05/runs/it_lora "
    "--adapter_alt chapter05/runs/it_qlora "
    "--test data/it_support/test.jsonl")

## 6. (Không cần train) Dùng adapter đã publish trên Hugging Face

In [ ]:
run('python -m chapter05.generate --base Qwen/Qwen3-4B-Instruct-2507 '
    '--adapter "bahree/ModelAdaptationBook#ch5-lora" --max_new_tokens 256 '
    '--prompt "How do I clear the DNS cache on Windows 11?"')
run('python -m chapter05.scripts.listing_5_3_evaluate --base Qwen/Qwen3-4B-Instruct-2507 '
    '--adapter "bahree/ModelAdaptationBook#ch5-lora" --test data/it_support/test.jsonl '
    '--out chapter05/runs/eval_report_hub')

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["chapter05/runs"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)